In [1]:
import duckdb
con = duckdb.connect('C:\\Users\\melvi\\Documents\\OpenF1DataWarehouse\\openf1_warehouse.duckdb')

In [8]:
# Viewing dim_meetings
print(con.execute ("""
                   SELECT *
                   FROM main.dim_meetings
                   LIMIT 10 """).fetchdf())

   meeting_key  year   country_name country_code circuit_short_name  \
0         1140  2023        Bahrain          BRN             Sakhir   
1         1141  2023        Bahrain          BRN             Sakhir   
2         1142  2023   Saudi Arabia          KSA             Jeddah   
3         1143  2023      Australia          AUS          Melbourne   
4         1207  2023     Azerbaijan          AZE               Baku   
5         1208  2023  United States          USA              Miami   
6         1209  2023          Italy          ITA              Imola   
7         1210  2023         Monaco          MON        Monte Carlo   
8         1211  2023          Spain          ESP          Catalunya   
9         1212  2023         Canada          CAN           Montreal   

         circuit_type date_start  
0           Permanent 2023-02-23  
1           Permanent 2023-03-03  
2  Temporary - Street 2023-03-17  
3  Temporary - Street 2023-03-31  
4  Temporary - Street 2023-04-28  
5  Tempo

In [13]:
# Viewing dim_sessions
print(con.execute ("""
                   SELECT *
                   FROM main.dim_sessions
                   ORDER BY session_key ASC LIMIT 10 """).fetchdf())

   session_key  meeting_key session_name session_type date_start   date_end
0         7763         1140        Day 2     Practice 2023-02-24 2023-02-24
1         7764         1140        Day 3     Practice 2023-02-25 2023-02-25
2         7765         1141   Practice 1     Practice 2023-03-03 2023-03-03
3         7766         1141   Practice 2     Practice 2023-03-03 2023-03-03
4         7767         1141   Practice 3     Practice 2023-03-04 2023-03-04
5         7768         1141   Qualifying   Qualifying 2023-03-04 2023-03-04
6         7772         1142   Practice 1     Practice 2023-03-17 2023-03-17
7         7773         1142   Practice 2     Practice 2023-03-17 2023-03-17
8         7774         1142   Practice 3     Practice 2023-03-18 2023-03-18
9         7775         1142   Qualifying   Qualifying 2023-03-18 2023-03-18


In [14]:
# Viewing dim_drivers
print(con.execute ("""
                   SELECT *
                   FROM main.dim_drivers
                   LIMIT 10 """).fetchdf())

   driver_number  session_key name_acronym        full_name        team_name  \
0              1         7763          VER   Max VERSTAPPEN  Red Bull Racing   
1              2         7763          SAR   Logan SARGEANT         Williams   
2              4         7763          NOR     Lando NORRIS          McLaren   
3             10         7763          GAS     Pierre GASLY           Alpine   
4             11         7763          PER     Sergio PEREZ  Red Bull Racing   
5             14         7763          ALO  Fernando ALONSO     Aston Martin   
6             16         7763          LEC  Charles LECLERC          Ferrari   
7             20         7763          MAG  Kevin MAGNUSSEN     Haas F1 Team   
8             21         7763          DEV    Nyck DE VRIES       AlphaTauri   
9             22         7763          TSU     Yuki TSUNODA       AlphaTauri   

  team_colour country_code  
0      3671C6          NED  
1      37BEDD          USA  
2      F58020          GBR  
3  

In [24]:
# Viewing fact_session_result
print(con.execute ("""
                   SELECT * 
                   FROM main.fact_session_result
                   LIMIT 25 """).fetchdf())

    session_key  driver_number  position duration  points    dnf    dns    dsq
0          7763             24         1    91.61     NaN  False  False  False
1          7763              1         2    91.65     NaN  False  False  False
2          7763             14         3   92.205     NaN  False  False  False
3          7763             21         4   92.222     NaN  False  False  False
4          7763             27         5   92.466     NaN  False  False  False
5          7763             55         6   92.486     NaN  False  False  False
6          7763              2         7   92.549     NaN  False  False  False
7          7763             16         8   92.725     NaN  False  False  False
8          7763             81         9   93.175     NaN  False  False  False
9          7763             10        10   93.186     NaN  False  False  False
10         7763             20        11   93.442     NaN  False  False  False
11         7763             31        12    93.49   

In [27]:
# Viewing fact_race_control
print(con.execute ("""
                   SELECT *
                   FROM main.fact_race_control
                   LIMIT 10 """).fetchdf())

   session_key  driver_number  lap_number  qualifying_phase       category  \
0         7763            NaN         NaN               NaN          Other   
1         7763            NaN         NaN               NaN            Drs   
2         7763            NaN         NaN               NaN           Flag   
3         7763            NaN         NaN               NaN  SessionStatus   
4         7763            NaN         NaN               NaN           Flag   
5         7763            NaN         NaN               NaN           Flag   
6         7763            NaN         NaN               NaN      SafetyCar   
7         7763            NaN         NaN               NaN           Flag   
8         7763            NaN         NaN               NaN  SessionStatus   
9         7763            NaN         NaN               NaN           Flag   

            flag   scope                              message  
0           None    None  RISK OF RAIN FOR THIS SESSION IS 0%  
1           N

In [ ]:
# TEST QUERY 1 - Number of Black and White flags per driver in a session
print(con.execute ("""
                   SELECT
                        session_key,
                        driver_number,
                        COUNT(flag) AS black_and_white_flags
                   FROM main.fact_race_control
                   WHERE driver_number IS NOT NULL
                        AND flag = 'BLACK AND WHITE'
                   GROUP BY session_key, driver_number
                   ORDER BY session_key ASC """).fetchdf())

     session_key  driver_number  black_and_white_flags
0           7772            4.0                      1
1           7773           21.0                      1
2           7774           14.0                      1
3           7774           22.0                      1
4           7779           55.0                      1
..           ...            ...                    ...
119        10014            1.0                      1
120        10014            6.0                      1
121        10022            5.0                      1
122        11249           44.0                      1
123        11253           41.0                      1

[124 rows x 3 columns]


In [ ]:
# TEST QUERY 2 - Joining fact_session_result with fact_race_control for hypothesis.
# Aims to see which final_position had 'x' amount of Black and White flags.
print(con.execute ("""
                   SELECT 
                        s.position AS final_position,
                        COUNT(r.flag) AS BW_flag_count
                   FROM main.fact_session_result s
                   LEFT JOIN main.fact_race_control r ON s.session_key = r.session_key
                   WHERE r.flag LIKE '%BLACK%'
                        AND s.position = 1
                   GROUP BY s.position
                   ORDER BY final_position, BW_flag_count DESC LIMIT 30 """).fetchdf())

   final_position  BW_flag_count
0               1            124


In [25]:
# TEST QUERY 3 - Number of black flags across all sessions
print(con.execute ("""
                   SELECT s.session_key, s.position, r.flag
                   FROM main.fact_session_result s
                   LEFT JOIN main.fact_race_control r ON s.session_key = r.session_key
                   WHERE r.flag = 'BLACK AND WHITE'
                        AND s.position = 1
                   GROUP BY s.session_key, s.position, r.flag """).fetchdf())

    session_key  position             flag
0          7953         1  BLACK AND WHITE
1          7779         1  BLACK AND WHITE
2          9591         1  BLACK AND WHITE
3          9607         1  BLACK AND WHITE
4          9617         1  BLACK AND WHITE
5          9102         1  BLACK AND WHITE
6          9110         1  BLACK AND WHITE
7          9473         1  BLACK AND WHITE
8          9476         1  BLACK AND WHITE
9          7772         1  BLACK AND WHITE
10         9152         1  BLACK AND WHITE
11         9072         1  BLACK AND WHITE
12         9474         1  BLACK AND WHITE
13         9905         1  BLACK AND WHITE
14         9888         1  BLACK AND WHITE
15         9942         1  BLACK AND WHITE
16         9183         1  BLACK AND WHITE
17         9637         1  BLACK AND WHITE
18         9106         1  BLACK AND WHITE
19         9141         1  BLACK AND WHITE
20         9165         1  BLACK AND WHITE
21         9550         1  BLACK AND WHITE
22         

In [41]:
con.close()